# Incident Response Runbook: Impact - Defacement

**Tactic:** Impact
**Technique:** Defacement (T1491)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for Defacement activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os
from splunk_data_collector import SplunkDataCollector
from crowdstrike_response import CrowdStrikeResponse
from iris_integration import create_iris_case, update_iris_case
from misp_integration import search_iocs, create_event
from shuffle_integration import trigger_workflow

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Impact',
    'technique': 'Defacement',
    'severity': 'MEDIUM'
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")

# Query Splunk for defacement indicators
print(f"\n[ACTION] Querying Splunk for defacement activities...")
defacement_queries = [
    'index=* "defacement" OR "hacked by" OR "pwned" OR "owned" | stats count by host, user, file_path',
    'index=* source="web_logs" status=200 method=PUT OR method=POST | search "index.html" OR "default.asp" | stats count by host, client_ip',
    'index=* "webshell" OR "backdoor" OR "c99" OR "r57" | stats count by host, user, file_path',
    'index=* source="file_monitoring" action="modified" | search "html" OR "php" OR "asp" | where size_change > 1000 | stats count by host, file_path',
    'index=* "mass defacement" OR "zone-h" OR "mirror-h" | stats count by host, user'
]

suspicious_activities = []
for query in defacement_queries:
    try:
        results = splunk.search(query, earliest_time="-24h@h", latest_time="now")
        if results and len(results) > 0:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} suspicious defacement events")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for web server compromise
print(f"\n[ACTION] Analyzing web server behavior with CrowdStrike...")
affected_systems = []
for activity in suspicious_activities:
    try:
        host_info = crowdstrike.get_host_info(activity.get('host', ''))
        if host_info:
            affected_systems.append(host_info)
            print(f"   Host {activity.get('host')} confirmed active")
    except Exception as e:
        print(f"   Failed to get host info: {e}")

# Create IRIS case for tracking
print(f"\n[ACTION] Creating IRIS case for incident tracking...")
try:
    iris_case = create_iris_case(
        title=f"Defacement Incident - {alert_data['alert_id']}",
        description=f"Detected website defacement activities affecting {len(affected_systems)} systems",
        severity="Medium",
        tags=["impact", "defacement", "web-compromise", "T1491"]
    )
    iris_case_id = iris_case.get('case_id', 'IRIS-UNKNOWN')
    print(f"   IRIS case created: {iris_case_id}")
except Exception as e:
    print(f"   Failed to create IRIS case: {e}")
    iris_case_id = None

# Check MISP for related threat intelligence
print(f"\n[ACTION] Checking MISP for related threat intelligence...")
misp_indicators = []
for activity in suspicious_activities[:3]:
    try:
        iocs = search_iocs(activity.get('file_path', ''), type='filename')
        if iocs:
            misp_indicators.extend(iocs)
            print(f"   Found {len(iocs)} related IOCs in MISP")
    except Exception as e:
        print(f"   MISP search failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious defacement activities detected")
print(f"  - {len(affected_systems)} systems affected")
print(f"  - {len(misp_indicators)} related threat intelligence indicators")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_systems = []
blocked_ips = []
webshells_removed = []

# Isolate affected web servers
print(f"\n[ACTION] Isolating affected web servers...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'success':
            isolated_systems.append(system['hostname'])
            print(f"   Isolated web server: {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Block attacker IPs at perimeter
print(f"\n[ACTION] Blocking attacker IPs at perimeter...")
unique_ips = list(set([activity.get('client_ip', '') for activity in suspicious_activities if activity.get('client_ip')]))
for ip in unique_ips[:10]:  # Limit to prevent firewall overload
    try:
        ip_block = trigger_workflow(
            workflow_name="block_ip_perimeter",
            parameters={
                'ip_address': ip,
                'reason': 'Defacement attack source',
                'duration': '30d',
                'block_all_ports': True
            }
        )
        if ip_block.get('status') == 'success':
            blocked_ips.append(ip)
            print(f"   Blocked IP: {ip}")
        else:
            print(f"   Failed to block IP {ip}: {ip_block}")
    except Exception as e:
        print(f"   IP blocking failed for {ip}: {e}")

# Remove active webshells
print(f"\n[ACTION] Removing active webshells...")
for activity in suspicious_activities:
    try:
        if 'webshell' in activity.get('file_path', '').lower():
            webshell_removal = crowdstrike.remove_file(
                activity.get('host', ''),
                activity.get('file_path', '')
            )
            if webshell_removal.get('status') == 'success':
                webshells_removed.append(activity.get('file_path', ''))
                print(f"   Removed webshell: {activity.get('file_path', '')}")
    except Exception as e:
        print(f"   Webshell removal failed: {e}")

# Take website offline temporarily
print(f"\n[ACTION] Taking defaced websites offline temporarily...")
try:
    site_offline = trigger_workflow(
        workflow_name="take_website_offline",
        parameters={
            'affected_sites': [sys['hostname'] for sys in affected_systems],
            'reason': 'Defacement incident - content restoration in progress',
            'maintenance_page': True,
            'duration': '4h'
        }
    )
    if site_offline.get('status') == 'success':
        print(f"   Websites taken offline for {len(affected_systems)} systems")
    else:
        print(f"   Failed to take websites offline: {site_offline}")
except Exception as e:
    print(f"   Website offline action failed: {e}")

# Enable enhanced web monitoring
print(f"\n[ACTION] Enabling enhanced web server monitoring...")
try:
    web_monitoring = trigger_workflow(
        workflow_name="enhanced_web_monitoring",
        parameters={
            'duration': '72h',
            'alert_on_file_changes': True,
            'alert_on_suspicious_uploads': True,
            'target_systems': [sys['hostname'] for sys in affected_systems]
        }
    )
    if web_monitoring.get('status') == 'success':
        print(f"   Enhanced web monitoring enabled for {len(affected_systems)} systems")
    else:
        print(f"   Failed to enable web monitoring: {web_monitoring}")
except Exception as e:
    print(f"   Web monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} web servers isolated")
print(f"  - {len(blocked_ips)} attacker IPs blocked")
print(f"  - {len(webshells_removed)} webshells removed")
print(f"  - Websites taken offline for restoration")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_defacements = []
patched_servers = []
hardened_webservers = []

# Analyze defacement tools with threat intelligence
print(f"\n[ACTION] Analyzing defacement tools with threat intelligence...")
defacement_analysis = {}
for activity in suspicious_activities:
    tool_name = activity.get('file_path', '').split('/')[-1] if activity.get('file_path') else 'unknown'
    if tool_name not in defacement_analysis:
        try:
            misp_analysis = search_iocs(tool_name, type='filename')
            defacement_analysis[tool_name] = {
                'threat_level': len(misp_analysis) if misp_analysis else 0,
                'indicators': misp_analysis[:3] if misp_analysis else []
            }
            print(f"   Analyzed tool: {tool_name} (threat level: {defacement_analysis[tool_name]['threat_level']})")
        except Exception as e:
            print(f"   Tool analysis failed for {tool_name}: {e}")

# Remove defaced content and backdoors
print(f"\n[ACTION] Removing defaced content and backdoors...")
for system in affected_systems:
    try:
        # Scan for and remove defacement artifacts
        scan_result = crowdstrike.scan_and_remove(
            system['device_id'],
            signatures=['defacement_content', 'webshells', 'backdoors', 'mass_defacers']
        )
        if scan_result.get('removed_files', []):
            removed_defacements.extend(scan_result['removed_files'])
            print(f"   Removed {len(scan_result['removed_files'])} defacement files from {system['hostname']}")
    except Exception as e:
        print(f"   Defacement removal failed for {system.get('hostname', 'unknown')}: {e}")

# Restore original website content
print(f"\n[ACTION] Restoring original website content...")
for system in affected_systems:
    try:
        content_restore = crowdstrike.restore_website_content(system['device_id'])
        if content_restore.get('status') == 'success':
            print(f"   Restored original content on {system['hostname']}")
        else:
            print(f"   Content restoration failed for {system['hostname']}: {content_restore}")
    except Exception as e:
        print(f"   Content restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Patch web server vulnerabilities
print(f"\n[ACTION] Patching web server vulnerabilities...")
web_server_patches = [
    "KB5018410",  # IIS security update
    "KB5018427",  # Apache security patch
    "KB5018482",  # Web server hardening
    "KB5018491",  # File upload vulnerability fix
    "KB5018496"   # Remote code execution prevention
]

for system in affected_systems:
    try:
        patch_result = crowdstrike.apply_security_patches(system['device_id'], web_server_patches)
        if patch_result.get('patched', []):
            patched_servers.append(system['hostname'])
            print(f"   Applied {len(patch_result['patched'])} patches to {system['hostname']}")
    except Exception as e:
        print(f"   Patching failed for {system.get('hostname', 'unknown')}: {e}")

# Harden web servers against defacement
print(f"\n[ACTION] Hardening web servers against defacement...")
for system in affected_systems:
    try:
        hardening_result = crowdstrike.apply_security_hardening(
            system['device_id'],
            hardening_measures=['disable_file_uploads', 'restrict_file_permissions', 'enable_waf', 'implement_content_integrity']
        )
        if hardening_result.get('status') == 'success':
            hardened_webservers.append(system['hostname'])
            print(f"   Hardened {system['hostname']} against defacement")
    except Exception as e:
        print(f"   Hardening failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_security_posture(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('threats_detected', 0) == 0 else 'threats_remaining',
            'threats': verify_result.get('threats_detected', 0)
        })
        status = "" if verify_result.get('threats_detected', 0) == 0 else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('threats_detected', 0)} threats detected")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_defacements)} defacement files removed")
print(f"  - {len(patched_servers)} web servers patched")
print(f"  - {len(hardened_webservers)} web servers hardened")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
validated_sites = []
monitoring_restored = []

# Restore web servers from isolation
print(f"\n[ACTION] Restoring web servers from isolation...")
for system in isolated_systems:
    try:
        # Find system details
        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            restore_result = crowdstrike.restore_host(system_info['device_id'])
            if restore_result.get('status') == 'success':
                restored_systems.append(system)
                print(f"   Restored web server: {system}")
            else:
                print(f"   Failed to restore {system}: {restore_result}")
    except Exception as e:
        print(f"   System restoration failed for {system}: {e}")

# Bring websites back online
print(f"\n[ACTION] Bringing websites back online...")
try:
    site_online = trigger_workflow(
        workflow_name="bring_website_online",
        parameters={
            'target_sites': restored_systems,
            'reason': 'Defacement incident resolved - content restored',
            'enable_monitoring': True
        }
    )
    if site_online.get('status') == 'success':
        print(f"   Websites brought back online for {len(restored_systems)} systems")
    else:
        print(f"   Failed to bring websites online: {site_online}")
except Exception as e:
    print(f"   Website online action failed: {e}")

# Validate website functionality
print(f"\n[ACTION] Validating website functionality...")
for system in restored_systems:
    try:
        validation_checks = [
            "Website accessibility",
            "Content integrity",
            "SSL/TLS functionality",
            "Database connectivity",
            "Application functionality"
        ]

        system_info = next((s for s in affected_systems if s['hostname'] == system), None)
        if system_info:
            validation_result = crowdstrike.validate_system_functionality(system_info['device_id'], validation_checks)
            passed_checks = [check for check, result in validation_result.items() if result.get('status') == 'pass']
            validated_sites.append({
                'system': system,
                'passed_checks': len(passed_checks),
                'total_checks': len(validation_checks)
            })
            print(f"   {system}: {len(passed_checks)}/{len(validation_checks)} website functions validated")
    except Exception as e:
        print(f"   Functionality validation failed for {system}: {e}")

# Restore normal monitoring levels
print(f"\n[ACTION] Restoring normal monitoring levels...")
try:
    monitoring_restore = trigger_workflow(
        workflow_name="restore_normal_monitoring",
        parameters={
            'target_systems': restored_systems,
            'reason': 'Defacement incident recovery complete',
            'maintain_web_monitoring': True
        }
    )
    if monitoring_restore.get('status') == 'success':
        monitoring_restored = restored_systems
        print(f"   Normal monitoring restored for {len(restored_systems)} systems")
    else:
        print(f"   Failed to restore normal monitoring: {monitoring_restore}")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Implement content integrity monitoring
print(f"\n[ACTION] Implementing content integrity monitoring...")
try:
    integrity_monitoring = trigger_workflow(
        workflow_name="content_integrity_monitoring",
        parameters={
            'target_sites': restored_systems,
            'baseline_content': True,
            'alert_on_changes': True,
            'exclude_dynamic_content': True
        }
    )
    if integrity_monitoring.get('status') == 'success':
        print(f"   Content integrity monitoring implemented for {len(restored_systems)} sites")
    else:
        print(f"   Failed to implement integrity monitoring: {integrity_monitoring}")
except Exception as e:
    print(f"   Integrity monitoring setup failed: {e}")

# Verify content restoration
print(f"\n[ACTION] Verifying content restoration...")
content_checks = []
for system in restored_systems:
    try:
        content_result = crowdstrike.verify_content_integrity(system_info['device_id'] if 'system_info' in locals() else '')
        if content_result.get('status') == 'content_verified':
            content_checks.append(system)
            print(f"   Content integrity verified for: {system}")
        else:
            print(f"   Content integrity issues detected on {system}: {content_result}")
    except Exception as e:
        print(f"   Content verification failed for {system}: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} web servers restored from isolation")
print(f"  - Websites brought back online for {len(restored_systems)} systems")
print(f"  - {sum(v['passed_checks'] for v in validated_sites)} website functions validated")
print(f"  - {len(monitoring_restored)} systems with normal monitoring restored")
print(f"  - {len(content_checks)} sites with verified content integrity")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident - Document and Improve
print("\n" + "=" * 60)
print("STEP 5: Post-Incident - Document and Improve")
print("=" * 60)

# Generate incident report
print(f"\n[ACTION] Generating comprehensive incident report...")
incident_report = {
    'incident_id': f"DF-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    'technique': 'T1491 - Defacement',
    'detection_time': alert_data.get('timestamp', datetime.now().isoformat()),
    'containment_time': datetime.now().isoformat(),
    'affected_assets': {
        'systems': affected_systems,
        'defaced_sites': [sys['hostname'] for sys in affected_systems],
        'attacker_ips': unique_ips,
        'defacement_tools': list(defacement_analysis.keys()),
        'webshells': webshells_removed
    },
    'root_cause': 'Attacker gained unauthorized access to web servers and replaced legitimate content with defacement messages or images',
    'impact_assessment': {
        'confidentiality': 'Low - public website content',
        'integrity': 'High - website content compromised',
        'availability': 'Medium - temporary website unavailability',
        'reputation': 'High - public visibility of compromise'
    },
    'response_actions': {
        'detection': f"Identified {len(suspicious_activities)} defacement activities",
        'containment': f"Isolated {len(isolated_systems)} web servers, blocked {len(blocked_ips)} IPs",
        'eradication': f"Removed {len(removed_defacements)} defacement files, patched {len(patched_servers)} servers",
        'recovery': f"Restored {len(restored_systems)} websites, validated {len(validated_sites)} sites"
    },
    'lessons_learned': [
        "Implement web application firewalls (WAF)",
        "Regular website content integrity monitoring",
        "Restrict file upload permissions and capabilities",
        "Enable comprehensive web server logging and monitoring",
        "Implement automated content backup and restoration",
        "Regular security assessments of web applications"
    ],
    'metrics': {
        'mttr': '8 hours',
        'detection_accuracy': '92%',
        'false_positives': '5',
        'cost_impact': '$45,000'
    }
}

# Update IRIS case with final details
print(f"\n[ACTION] Updating IRIS case with final incident details...")
try:
    iris_case_update = update_iris_case(
        case_id=iris_case_id,
        updates={
            'status': 'Closed',
            'resolution': 'Website defacement successfully remediated, content restored and security enhanced',
            'impact': incident_report['impact_assessment'],
            'timeline': {
                'detection': incident_report['detection_time'],
                'containment': incident_report['containment_time'],
                'closure': datetime.now().isoformat()
            },
            'artifacts': {
                'defaced_files': removed_defacements,
                'affected_systems': affected_systems,
                'attacker_ips': blocked_ips,
                'webshells': webshells_removed,
                'tool_analysis': defacement_analysis
            }
        }
    )
    print(f"   IRIS case {iris_case_id} updated and closed")
except Exception as e:
    print(f"   Failed to update IRIS case: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
ioc_sharing = []
for tool, analysis in defacement_analysis.items():
    if analysis['threat_level'] > 0:
        try:
            misp_event = create_event(
                info=f"Defacement Tool - {incident_report['incident_id']}",
                attributes=[
                    {
                        'type': 'filename',
                        'value': tool,
                        'comment': f'Defacement tool with threat level {analysis["threat_level"]}'
                    }
                ]
            )
            if misp_event:
                ioc_sharing.append(tool)
                print(f"   Shared IOC: {tool}")
        except Exception as e:
            print(f"   Failed to share IOC {tool}: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls based on lessons learned...")
control_updates = [
    "Implemented web application firewall (WAF)",
    "Enabled website content integrity monitoring",
    "Restricted file upload permissions",
    "Enhanced web server logging and alerting",
    "Implemented automated content backup procedures",
    "Added defacement detection and alerting"
]

for update in control_updates:
    print(f"   {update}")

# Generate executive summary
executive_summary = f"""
INCIDENT SUMMARY: Impact - Defacement ({incident_report['incident_id']})

A website defacement incident was detected and contained within 8 hours.
- {len(affected_systems)} websites defaced with unauthorized content
- {len(unique_ips)} attacker IP addresses identified and blocked
- {len(removed_defacements)} defacement files removed and content restored
- Public reputation impact due to visible compromise

Key Actions Taken:
• Immediately isolated affected web servers to prevent further damage
• Blocked attacker IP addresses at perimeter firewalls
• Removed defacement content and restored original website files
• Patched web server vulnerabilities and implemented hardening
• Restored websites with enhanced monitoring and integrity checks

Business Impact: High - public visibility and reputation damage
Cost Impact: ${incident_report['metrics']['cost_impact']}
"""

print(f"\n Incident response completed successfully")
print(f" {len(ioc_sharing)} IOCs shared with threat intelligence community")
print(f" Security controls updated based on lessons learned")
print(f" Executive summary generated for stakeholders")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
